In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import urllib.request
import zipfile
import io

catalog = "dbr_dev"
schema  = "live_transit_monitor"
volume  = "project_volume"

gtfs_url    = "https://ckan.multimediagdansk.pl/dataset/c24aa637-3619-4dc2-a171-a23eec8f2172/resource/30e783e4-2bec-4a7d-bb22-ee3e3b26ca96/download/gtfsgoogle.zip"
batch_path  = f"/Volumes/{catalog}/{schema}/{volume}/batch"
gtfs_path   = f"{batch_path}/gtfs"   

gtfs_config = {
    "routes.txt":         ("bronze_gtfs_routes",         ["route_id"]),
    "stops.txt":          ("bronze_gtfs_stops",          ["stop_id"]),
    "trips.txt":          ("bronze_gtfs_trips",          ["trip_id"]),
    "stop_times.txt":     ("bronze_gtfs_stop_times",     ["trip_id", "stop_sequence"]),
    "calendar_dates.txt": ("bronze_gtfs_calendar_dates", ["service_id", "date"]),
    "agency.txt":         ("bronze_gtfs_agency",         ["agency_id"]),
}

print(f"Location: {catalog}.{schema}.bronze_gtfs_*")
print(f"GTFS files location: {gtfs_path}")

In [0]:
with urllib.request.urlopen(gtfs_url) as resp:
    zip_bytes = resp.read()
print(f"Downloaded {len(zip_bytes):,} bytes.")

zf = zipfile.ZipFile(io.BytesIO(zip_bytes))
for fname in gtfs_config.keys():
    content = zf.read(fname)
    dest = f"{gtfs_path}/{fname}"
    dbutils.fs.put(dest, content.decode("utf-8"), overwrite=True)
    print(f"Saved {fname} -> {dest} ({len(content):,} bytes)")

print("\nUnpack GTFS to volume.")

In [0]:
def load_gtfs_file(filename, table_name, merge_keys):

    full_table = f"{catalog}.{schema}.{table_name}"
    src_path   = f"{gtfs_path}/{filename}"

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(src_path)
        .withColumn("source_file", F.lit(filename))
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("load_date", F.current_date())
    )

    if not spark.catalog.tableExists(full_table):
        df.write.format("delta").saveAsTable(full_table)
        action = "created"
    else:
        delta_tbl = DeltaTable.forName(spark, full_table)
        cond = " AND ".join([f"t.`{k}` = s.`{k}`" for k in merge_keys])
        (delta_tbl.alias("t")
            .merge(df.alias("s"), cond)
            .whenNotMatchedInsertAll()
            .execute())
        action = "merged"

    count = spark.table(full_table).count()
    print(f"  {table_name}: {action}, {count:,} rows (key: {', '.join(merge_keys)})")

In [0]:
for filename, (table_name, merge_keys) in gtfs_config.items():
    load_gtfs_file(filename, table_name, merge_keys)
print("\nAll files GTFS loaded.")

In [0]:
print("Summary of GTFS tables:\n")
for filename, (table_name, merge_keys) in gtfs_config.items():
    full = f"{catalog}.{schema}.{table_name}"
    cnt = spark.table(full).count()
    print(f"{table_name}: {cnt:,} rows")

# Checking columns and data
display(spark.sql(f"SELECT * FROM {catalog}.{schema}.bronze_gtfs_routes LIMIT 5"))

In [0]:
display(spark.table("dbr_dev.live_transit_monitor.bronze_gtfs_stops"))

In [0]:
display(spark.sql("""
    SELECT stop_id, stop_name, stop_lat, stop_lon, stop_code
    FROM dbr_dev.live_transit_monitor.bronze_gtfs_stops
    LIMIT 20
"""))